In [0]:
# Create the customer_complaint_escalation_agent (main) catalog
spark.sql("CREATE CATALOG IF NOT EXISTS main")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.default")

Create predictions table to store the multi-agent workflow's final predictions, including the RCA and response.

In [0]:
spark.sql("""
DROP TABLE IF EXISTS main.default.predictions
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS main.default.predictions (
  complaint_id STRING,
  assigned_department STRING,
  root_cause_analysis STRING,
  recommended_actions STRING,
  priority_level STRING,
  customer_email_subject STRING,
  customer_email_body STRING
)
""")

Create customer_transactions table to verify customer transactions and late fees

In [0]:
spark.sql("""
DROP TABLE IF EXISTS main.default.customer_transactions
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS main.default.customer_transactions (
  customer_id INTEGER,
  vendor STRING,
  memo STRING,
  transaction_type STRING,
  date DATE,
  amount DOUBLE
)
""")

Seed the customer_transactions table with some data for puchases, late payment fees, and interest charges.

In [0]:
import pandas as pd
from pyspark.sql.functions import col

spark_df = spark.createDataFrame(pd.DataFrame(
    [
        [1, "Visa", "Visa Purchase", "debit", "2026-06-01", 100],
        [1, "Visa", "Visa Purchase", "debit", "2026-06-02", 101],
        [1, "Visa", "Visa Purchase", "debit", "2026-06-03", 102],
        [1, "Visa", "Visa Purchase", "debit", "2026-06-03", 102],
        [2, "Visa", "Visa Purchase", "debit", "2026-06-01", 0.01],
        [3, "Employer", "Payroll", "credit", "2026-06-01", 5000],
        [4, "Visa", "Visa Purchase", "debit", "2026-06-01", 100],
        [4, "Visa", "Visa Purchase", "debit", "2026-06-02", 101],
        [4, "Visa", "Visa Purchase", "debit", "2026-06-03", 102],
        [4, "Visa", "Visa Purchase", "debit", "2026-06-01", 0.01],
        [4, "Bank", "Late Payment Fee", "debit", "2026-06-01", 35]
    ],
    columns=["customer_id", "vendor", "memo", "transaction_type", "date", "amount"],
))
spark_df = spark_df.select(
    col("customer_id").cast("int"),
    col("vendor"),
    col("memo"),
    col("transaction_type"),
    col("date").cast("date"),
    col("amount")
)
spark_df.write.mode("overwrite").saveAsTable("main.default.customer_transactions")

display(spark.table("main.default.customer_transactions").toPandas().head(11))

Create SQL UC Function to query transactions by customer_id

In [0]:
%sql
CREATE OR REPLACE FUNCTION main.default.get_customer_transactions(customer_id INT COMMENT 'The integer id of the customer')
RETURNS TABLE (
  customer_id INT,
  vendor STRING,
  memo STRING,
  transaction_type STRING,
  date DATE,
  amount DOUBLE
)
COMMENT 'Returns all transaction records for the specified customer_id'
RETURN
SELECT
  customer_id,
  vendor,
  memo,
  transaction_type,
  date,
  amount
FROM main.default.customer_transactions
WHERE customer_id = get_customer_transactions.customer_id;